In [1]:
%load_ext autoreload
%autoreload 2
import sys
import os
project_root = os.path.abspath("")
if project_root not in sys.path:
    sys.path.append(project_root)
import sleap
from pathlib import Path
from ipywidgets import widgets
from IPython.display import display

%matplotlib widget

In [3]:
import sleap
import pandas as pd
from pathlib import Path
import re

SUBJ_ID = "038"
DATE = "20251029"
BASE_DIR = Path("/Volumes/harris/hypnose")

deriv_dir = BASE_DIR / "derivatives"
subj_pattern = f"sub-{SUBJ_ID}_*"
subj_dirs = list(deriv_dir.glob(subj_pattern))

if not subj_dirs:
    raise FileNotFoundError(f"No subject directory found matching: {subj_pattern}")

subj_dir = subj_dirs[0]

# Find the session directory
session_pattern = f"ses-*_date-{DATE}"
session_dirs = list(subj_dir.glob(session_pattern))

if not session_dirs:
    raise FileNotFoundError(f"No session found matching: {session_pattern}")

session_dir = session_dirs[0]
results_dir = session_dir / "saved_analysis_results"

if not results_dir.exists():
    raise FileNotFoundError(f"Results directory not found: {results_dir}")

# Find all .predictions.slp files and sort chronologically
slp_files = sorted(results_dir.glob("VideoData_*.predictions.slp"))

if not slp_files:
    raise FileNotFoundError(f"No .slp files found in {results_dir}")

print(f"Found {len(slp_files)} video(s) to process:")
for i, f in enumerate(slp_files, 1):
    print(f"  {i}. {f.name}")

# Process each video
for video_number, slp_path in enumerate(slp_files, 1):
    output_path = results_dir / f"sleap_tracking_video{video_number}.csv"
    
    print(f"\n[{video_number}/{len(slp_files)}] Processing: {slp_path.name}")
    
    # Load predictions
    labels = sleap.load_file(str(slp_path))
    
    # Extract all node positions for all frames
    data = []
    for lf in labels:
        frame_idx = lf.frame_idx
        for instance_idx, instance in enumerate(lf.instances):
            row = {"frame": frame_idx, "instance": instance_idx}
            for node, point in instance.nodes_points:
                row[f"{node.name}_x"] = point.x
                row[f"{node.name}_y"] = point.y
                row[f"{node.name}_score"] = point.score
            data.append(row)
    
    df = pd.DataFrame(data)
    
    # Calculate centroid from core nodes
    core_nodes = ['right_ear', 'left_ear', 'center_head', 'neck', 'center', 'center_back']
    centroid_x_cols = [f"{node}_x" for node in core_nodes if f"{node}_x" in df.columns]
    centroid_y_cols = [f"{node}_y" for node in core_nodes if f"{node}_y" in df.columns]
    centroid_score_cols = [f"{node}_score" for node in core_nodes if f"{node}_score" in df.columns]
    
    df['centroid_x'] = df[centroid_x_cols].mean(axis=1)
    df['centroid_y'] = df[centroid_y_cols].mean(axis=1)
    df['centroid_score'] = df[centroid_score_cols].mean(axis=1)
    
    # Save to CSV
    df.to_csv(output_path, index=False)
    
    # Summary output
    print(f"  ✓ Saved to: {output_path.name}")
    print(f"    Total frames: {len(df)}")
    print(f"    Frame range: {int(df['frame'].min())} - {int(df['frame'].max())}")
    print(f"    Valid centroid frames: {df['centroid_x'].notna().sum()}")
    print(f"    Centroid score range: {df['centroid_score'].min():.4f} - {df['centroid_score'].max():.4f}")

print(f"\n✅ All videos processed!")


Found 3 video(s) to process:
  1. VideoData_1904-01-02T03-00-00.predictions.slp
  2. VideoData_1904-01-02T04-00-00.predictions.slp
  3. VideoData_1904-01-02T05-00-00.predictions.slp

[1/3] Processing: VideoData_1904-01-02T03-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video1.csv
    Total frames: 177154
    Frame range: 0 - 197941
    Valid centroid frames: 177154
    Centroid score range: 0.2003 - 1.1570

[2/3] Processing: VideoData_1904-01-02T04-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video2.csv
    Total frames: 162883
    Frame range: 0 - 215805
    Valid centroid frames: 162883
    Centroid score range: 0.2001 - 1.1490

[3/3] Processing: VideoData_1904-01-02T05-00-00.predictions.slp
  ✓ Saved to: sleap_tracking_video3.csv
    Total frames: 76733
    Frame range: 163 - 101737
    Valid centroid frames: 76733
    Centroid score range: 0.2004 - 1.1260

✅ All videos processed!
